# Multi-Agent Introduction

Adding a second agent to an environment breaks a core assumption of single-agent RL:
the environment's dynamics are stationary. This notebook builds intuition for why
multi-agent problems are harder and walks through a competitive two-agent gridworld
with self-play.

In [ ]:
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

## Why Multi-Agent Changes the Problem

In single-agent RL, the environment obeys fixed rules and the agent learns to navigate them.
The optimal policy, once found, stays optimal.

In a multi-agent setting, the other agents are also learning.
What works against a random opponent often fails against a trained one.
And what works against a trained opponent may fail if the opponent updates its strategy.

This **non-stationarity** means:
- Training is less stable: the "environment" shifts under your feet
- A policy that reaches 90% win rate may collapse after the opponent adapts
- Evaluation against a fixed baseline (a frozen old policy) is the standard approach

The two-player competitive setting also introduces a key distinction:
- **Zero-sum**: one agent's gain is the other's loss (Chess, Go, the game below)
- **Cooperative**: agents must work together to maximize shared reward
- **Mixed**: elements of both (most real environments)

## Competitive GridWorld Environment

Two agents share a 7x7 grid. A single goal tile exists. The agent that
reaches the goal first wins and receives +1; the other receives -1.
Each agent can move in four directions or stay still. If both reach the
goal on the same step, it's a draw.

Agents cannot occupy the same cell (they block each other).

In [ ]:
GRID_SIZE = 7
ACTIONS = [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]  # stay, up, down, left, right
ACTION_NAMES = ["stay", "up", "down", "left", "right"]


class CompetitiveGridWorld:
    def __init__(self, size=GRID_SIZE, goal=(3, 6), walls=None):
        self.size = size
        self.goal = goal
        self.walls = set(walls or [])

    def reset(self):
        """Return initial positions: agent 0 at top-left, agent 1 at bottom-left."""
        return (0, 0), (self.size - 1, 0)

    def _clamp(self, pos, delta):
        r, c = pos[0] + delta[0], pos[1] + delta[1]
        r = max(0, min(self.size - 1, r))
        c = max(0, min(self.size - 1, c))
        if (r, c) in self.walls:
            return pos  # stay in place if wall
        return (r, c)

    def step(self, pos0, pos1, action0, action1):
        """
        Returns new_pos0, new_pos1, reward0, reward1, done.
        reward: +1 win, -1 loss, 0 ongoing or draw.
        """
        new0 = self._clamp(pos0, ACTIONS[action0])
        new1 = self._clamp(pos1, ACTIONS[action1])

        # Prevent occupying the same cell
        if new0 == new1:
            new0, new1 = pos0, pos1  # both stay

        at_goal0 = new0 == self.goal
        at_goal1 = new1 == self.goal

        if at_goal0 and at_goal1:
            return new0, new1, 0, 0, True   # draw
        if at_goal0:
            return new0, new1, 1, -1, True  # agent 0 wins
        if at_goal1:
            return new0, new1, -1, 1, True  # agent 1 wins
        return new0, new1, 0, 0, False

    def visualize(self, pos0, pos1, title=""):
        display = np.ones((self.size, self.size, 3), dtype=np.uint8) * 240
        for r, c in self.walls:
            display[r, c] = [60, 60, 60]
        gr, gc = self.goal
        display[gr, gc] = [255, 215, 0]   # goal: gold
        display[pos0[0], pos0[1]] = [50, 150, 255]  # agent 0: blue
        display[pos1[0], pos1[1]] = [255, 80, 80]   # agent 1: red
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(display, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


env = CompetitiveGridWorld()
p0, p1 = env.reset()
env.visualize(p0, p1, title="Initial state (blue=agent0, red=agent1, gold=goal)")

## Defining Policies

We'll compare three policies:
1. **Random**: choose a uniformly random action each step
2. **Greedy**: always move toward the goal (Manhattan distance minimizing)
3. **Blocking greedy**: move toward the goal, but if the opponent is closer, try to block

In [ ]:
def random_policy(my_pos, other_pos, goal):
    return random.randint(0, len(ACTIONS) - 1)


def greedy_policy(my_pos, other_pos, goal):
    """Move in the direction that reduces Manhattan distance to goal."""
    best_action = 0
    best_dist = abs(my_pos[0] - goal[0]) + abs(my_pos[1] - goal[1])
    for i, (dr, dc) in enumerate(ACTIONS[1:], start=1):
        nr = max(0, min(GRID_SIZE - 1, my_pos[0] + dr))
        nc = max(0, min(GRID_SIZE - 1, my_pos[1] + dc))
        dist = abs(nr - goal[0]) + abs(nc - goal[1])
        if dist < best_dist:
            best_dist = dist
            best_action = i
    return best_action


def blocking_greedy_policy(my_pos, other_pos, goal):
    """Greedy toward goal, but if opponent is ahead, move to intercept."""
    my_dist = abs(my_pos[0] - goal[0]) + abs(my_pos[1] - goal[1])
    opp_dist = abs(other_pos[0] - goal[0]) + abs(other_pos[1] - goal[1])

    if opp_dist < my_dist - 1:
        # Opponent is significantly closer; try to move toward the opponent
        best_action = 0
        best_dist = abs(my_pos[0] - other_pos[0]) + abs(my_pos[1] - other_pos[1])
        for i, (dr, dc) in enumerate(ACTIONS[1:], start=1):
            nr = max(0, min(GRID_SIZE - 1, my_pos[0] + dr))
            nc = max(0, min(GRID_SIZE - 1, my_pos[1] + dc))
            dist = abs(nr - other_pos[0]) + abs(nc - other_pos[1])
            if dist < best_dist:
                best_dist = dist
                best_action = i
        return best_action
    return greedy_policy(my_pos, other_pos, goal)

## Tournament: Random vs. Greedy

We run N games between two policies and track win rates.
Agent 0 starts at (0,0), Agent 1 starts at (6,0).

In [ ]:
def run_tournament(policy0, policy1, n_games=500, max_steps=100):
    wins = {"agent0": 0, "agent1": 0, "draw": 0, "timeout": 0}

    for _ in range(n_games):
        p0, p1 = env.reset()
        for _ in range(max_steps):
            a0 = policy0(p0, p1, env.goal)
            a1 = policy1(p1, p0, env.goal)
            p0, p1, r0, r1, done = env.step(p0, p1, a0, a1)
            if done:
                if r0 > 0:
                    wins["agent0"] += 1
                elif r1 > 0:
                    wins["agent1"] += 1
                else:
                    wins["draw"] += 1
                break
        else:
            wins["timeout"] += 1

    total = n_games
    for k, v in wins.items():
        print(f"  {k}: {v} ({v/total*100:.1f}%)")
    return wins


print("Random (agent0) vs. Greedy (agent1):")
run_tournament(random_policy, greedy_policy)

print()
print("Greedy (agent0) vs. Greedy (agent1):")
run_tournament(greedy_policy, greedy_policy)

print()
print("Greedy (agent0) vs. Blocking Greedy (agent1):")
run_tournament(greedy_policy, blocking_greedy_policy)

## Self-Play

Self-play trains an agent by having it compete against a copy of itself.
Periodically, the opponent is updated to a checkpoint of the current policy.
This produces a curriculum: the opponent always provides a challenge at the current skill level.

We'll implement a simple tabular Q-learning agent and train it via self-play.
The Q-table maps (my_pos, other_pos) to action values.

Note: full self-play at scale requires careful implementation (league training, population
diversity) to avoid converging to exploitable strategies. This is a minimal demonstration.

In [ ]:
class QAgent:
    def __init__(self, alpha=0.1, gamma=0.95, epsilon=0.2):
        self.q = defaultdict(lambda: np.zeros(len(ACTIONS)))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def act(self, my_pos, other_pos, greedy=False):
        state = (my_pos, other_pos)
        if not greedy and random.random() < self.epsilon:
            return random.randint(0, len(ACTIONS) - 1)
        return int(np.argmax(self.q[state]))

    def update(self, my_pos, other_pos, action, reward, next_my_pos, next_other_pos, done):
        state = (my_pos, other_pos)
        next_state = (next_my_pos, next_other_pos)
        target = reward if done else reward + self.gamma * np.max(self.q[next_state])
        self.q[state][action] += self.alpha * (target - self.q[state][action])

    def copy_weights(self, other_agent):
        self.q = defaultdict(lambda: np.zeros(len(ACTIONS)),
                             {k: v.copy() for k, v in other_agent.q.items()})


def selfplay_training(n_episodes=3000, opponent_update_interval=300):
    agent = QAgent()
    opponent = QAgent(epsilon=0.0)  # frozen opponent (greedy)

    win_rates = []
    window = 100
    recent_wins = []

    for ep in range(n_episodes):
        p0, p1 = env.reset()
        for _ in range(80):
            a0 = agent.act(p0, p1)
            a1 = opponent.act(p1, p0)  # opponent plays from its own perspective
            new_p0, new_p1, r0, r1, done = env.step(p0, p1, a0, a1)
            agent.update(p0, p1, a0, r0, new_p0, new_p1, done)
            p0, p1 = new_p0, new_p1
            if done:
                recent_wins.append(1 if r0 > 0 else 0)
                break
        else:
            recent_wins.append(0)

        if len(recent_wins) > window:
            recent_wins.pop(0)

        if (ep + 1) % window == 0:
            wr = sum(recent_wins) / len(recent_wins)
            win_rates.append(wr)

        # Update the frozen opponent with current policy
        if (ep + 1) % opponent_update_interval == 0:
            opponent.copy_weights(agent)

    return agent, win_rates


print("Training Q-agent via self-play...")
trained_agent, win_rates = selfplay_training()
print("Done.")

plt.figure(figsize=(8, 4))
plt.plot(range(len(win_rates)), win_rates, marker="o", markersize=4)
plt.axhline(0.5, color="gray", linestyle="--", label="50% baseline")
plt.xlabel("Training episode (x100)")
plt.ylabel("Win rate (last 100 episodes)")
plt.title("Self-play Q-agent win rate vs. frozen opponent")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluating the Trained Agent

In [ ]:
def eval_agent_vs_policy(agent, policy, n_games=200):
    wins = {"trained": 0, "opponent": 0, "draw": 0}
    for _ in range(n_games):
        p0, p1 = env.reset()
        for _ in range(80):
            a0 = agent.act(p0, p1, greedy=True)
            a1 = policy(p1, p0, env.goal)
            p0, p1, r0, r1, done = env.step(p0, p1, a0, a1)
            if done:
                if r0 > 0:
                    wins["trained"] += 1
                elif r1 > 0:
                    wins["opponent"] += 1
                else:
                    wins["draw"] += 1
                break
    for k, v in wins.items():
        print(f"  {k}: {v/n_games*100:.1f}%")


print("Trained agent vs. Random:")
eval_agent_vs_policy(trained_agent, random_policy)

print()
print("Trained agent vs. Greedy:")
eval_agent_vs_policy(trained_agent, greedy_policy)

## Deep Multi-Agent Training with Cooperative Goals

The sections above use tabular Q-learning on a competitive environment.
Now we build a cooperative two-agent scenario: both agents must reach their own
distinct goal cells in a shared grid. We implement:

1. A cooperative two-agent GridWorld
2. Independent learners (separate policies, no communication)
3. Parameter sharing (one shared policy, agent ID as input)
4. A simple communication channel (broadcast vectors before acting)
5. Reward shaping for cooperation
6. A simplified QMIX monotonic mixing network
7. Full training metrics with per-agent and joint reward plots

In [ ]:
# pip install torch  (numpy and matplotlib already imported)
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## Cooperative Two-Agent GridWorld Environment

Two agents share a 6x6 grid. Each has a different goal cell. An episode ends when
both agents have reached their goals. Agents cannot occupy the same cell.

Observation for agent i: `[row_i, col_i, row_j, col_j]` (fully observable).

In [ ]:
COOP_SIZE = 6
COOP_ACTIONS = [(-1,0),(1,0),(0,-1),(0,1),(0,0)]   # up, down, left, right, stay
N_COOP_ACTIONS = len(COOP_ACTIONS)
OBS_DIM = 4   # [r0, c0, r1, c1] for each agent


class CoopGridWorld:
    """
    Cooperative two-agent gridworld.

    Agent 0 starts at (0,0) and must reach goal_0 (bottom-right area).
    Agent 1 starts at (0, SIZE-1) and must reach goal_1 (bottom-left area).
    Both goals must be reached to end the episode.
    """

    def __init__(self, size=COOP_SIZE, goal_0=(5, 5), goal_1=(5, 0)):
        self.size    = size
        self.goal_0  = goal_0
        self.goal_1  = goal_1
        self._pos    = [None, None]
        self._at_goal = [False, False]

    def reset(self):
        self._pos     = [(0, 0), (0, self.size - 1)]
        self._at_goal = [False, False]
        return self._obs()

    def _obs(self):
        """Return (obs_agent0, obs_agent1) each as a list of 4 floats."""
        r0, c0 = self._pos[0]
        r1, c1 = self._pos[1]
        obs0 = [r0 / (self.size-1), c0 / (self.size-1),
                r1 / (self.size-1), c1 / (self.size-1)]
        obs1 = [r1 / (self.size-1), c1 / (self.size-1),
                r0 / (self.size-1), c0 / (self.size-1)]
        return obs0, obs1

    def _move(self, pos, action):
        dr, dc = COOP_ACTIONS[action]
        r = max(0, min(self.size - 1, pos[0] + dr))
        c = max(0, min(self.size - 1, pos[1] + dc))
        return (r, c)

    def step(self, action_0, action_1):
        """
        Returns obs_0, obs_1, reward_0, reward_1, done, info.
        """
        new0 = self._move(self._pos[0], action_0)
        new1 = self._move(self._pos[1], action_1)

        # Prevent collision: if both try to occupy same cell, they stay
        if new0 == new1:
            new0 = self._pos[0]
            new1 = self._pos[1]

        self._pos[0] = new0
        self._pos[1] = new1

        # Check goal arrival
        reached_0 = (new0 == self.goal_0)
        reached_1 = (new1 == self.goal_1)
        self._at_goal[0] = self._at_goal[0] or reached_0
        self._at_goal[1] = self._at_goal[1] or reached_1

        # Shaping: negative Manhattan distance to own goal
        d0 = abs(new0[0]-self.goal_0[0]) + abs(new0[1]-self.goal_0[1])
        d1 = abs(new1[0]-self.goal_1[0]) + abs(new1[1]-self.goal_1[1])
        rew_0 = -0.01 * d0
        rew_1 = -0.01 * d1

        if reached_0:
            rew_0 += 1.0
        if reached_1:
            rew_1 += 1.0

        done = bool(self._at_goal[0] and self._at_goal[1])
        obs0, obs1 = self._obs()
        return obs0, obs1, rew_0, rew_1, done, {"d0": d0, "d1": d1}

    def visualize(self, title=""):
        display = np.ones((self.size, self.size, 3), dtype=np.uint8) * 240
        gr0, gc0 = self.goal_0
        gr1, gc1 = self.goal_1
        display[gr0, gc0] = [80, 200, 80]    # goal 0: green
        display[gr1, gc1] = [200, 80, 80]    # goal 1: red
        r0, c0 = self._pos[0]
        r1, c1 = self._pos[1]
        display[r0, c0] = [50, 120, 255]     # agent 0: blue
        display[r1, c1] = [255, 160, 30]     # agent 1: orange
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(display, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        patches = [
            mpatches.Patch(color=[.2, .47, 1], label="Agent 0 (blue)"),
            mpatches.Patch(color=[1, .63, .12], label="Agent 1 (orange)"),
            mpatches.Patch(color=[.31, .78, .31], label="Goal 0 (green)"),
            mpatches.Patch(color=[.78, .31, .31], label="Goal 1 (red)"),
        ]
        ax.legend(handles=patches, fontsize=7, loc="upper right")
        plt.tight_layout()
        plt.show()


coop_env = CoopGridWorld()
obs0, obs1 = coop_env.reset()
coop_env.visualize("Initial state")
print("obs agent 0:", obs0)
print("obs agent 1:", obs1)

## Independent Learner Baseline

The simplest multi-agent approach: each agent trains its own separate policy
gradient network and treats the other agent as part of the environment.

No coordination, no communication. Training is unstable because as one agent
improves its policy, it changes the effective environment seen by the other.

In [ ]:
class PolicyMLP(nn.Module):
    """Simple policy MLP: obs -> action logits."""
    def __init__(self, obs_dim, hidden_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, x):
        return self.net(x)


def reinforce_update(policy, optimizer, states, actions, returns, gamma=0.99):
    """
    Simple REINFORCE update: gradient ascent on log pi(a|s) * G_t.
    states  : list of obs vectors
    actions : list of int
    returns : list of discounted returns
    """
    if not states:
        return 0.0
    obs_t   = torch.FloatTensor(states)
    act_t   = torch.LongTensor(actions)
    ret_t   = torch.FloatTensor(returns)
    ret_t   = (ret_t - ret_t.mean()) / (ret_t.std() + 1e-8)

    logits  = policy(obs_t)
    dist    = Categorical(logits=logits)
    log_p   = dist.log_prob(act_t)
    loss    = -(log_p * ret_t).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


def discounted_returns(rewards, gamma=0.99):
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns


def run_episode(env, policy_0, policy_1, max_steps=50, train=True,
                opt0=None, opt1=None):
    """Run one episode; optionally update both policies via REINFORCE."""
    obs0, obs1 = env.reset()
    tr0 = {"states": [], "actions": [], "rewards": []}
    tr1 = {"states": [], "actions": [], "rewards": []}

    for _ in range(max_steps):
        with torch.no_grad():
            a0 = Categorical(logits=policy_0(torch.FloatTensor(obs0))).sample().item()
            a1 = Categorical(logits=policy_1(torch.FloatTensor(obs1))).sample().item()

        next_obs0, next_obs1, r0, r1, done, _ = env.step(a0, a1)

        tr0["states"].append(obs0); tr0["actions"].append(a0); tr0["rewards"].append(r0)
        tr1["states"].append(obs1); tr1["actions"].append(a1); tr1["rewards"].append(r1)

        obs0, obs1 = next_obs0, next_obs1
        if done:
            break

    if train and opt0 and opt1:
        reinforce_update(policy_0, opt0, tr0["states"], tr0["actions"],
                         discounted_returns(tr0["rewards"]))
        reinforce_update(policy_1, opt1, tr1["states"], tr1["actions"],
                         discounted_returns(tr1["rewards"]))

    return sum(tr0["rewards"]), sum(tr1["rewards"])


# ----- Train independent learners -------------------------------------------
torch.manual_seed(0)
HIDDEN = 64
N_EPISODES = 400

policy_ind_0 = PolicyMLP(OBS_DIM, HIDDEN, N_COOP_ACTIONS)
policy_ind_1 = PolicyMLP(OBS_DIM, HIDDEN, N_COOP_ACTIONS)
opt_ind_0    = optim.Adam(policy_ind_0.parameters(), lr=1e-3)
opt_ind_1    = optim.Adam(policy_ind_1.parameters(), lr=1e-3)

ind_rewards_0, ind_rewards_1, ind_joint = [], [], []

env_coop = CoopGridWorld()
for ep in range(N_EPISODES):
    r0, r1 = run_episode(env_coop, policy_ind_0, policy_ind_1,
                         opt0=opt_ind_0, opt1=opt_ind_1)
    ind_rewards_0.append(r0)
    ind_rewards_1.append(r1)
    ind_joint.append(r0 + r1)

print(f"Independent learners -- last 50 ep joint reward: "
      f"{np.mean(ind_joint[-50:]):.3f}")

## Parameter Sharing

One shared policy network serves both agents. Each agent appends a one-hot
agent ID to its observation so the network can distinguish them.

Benefits: half the parameters, experiences from both agents update the same
network (effectively double the data), and the policy generalises across roles.

Observation becomes: `[r_self, c_self, r_other, c_other, id_0, id_1]`

In [ ]:
N_AGENTS = 2
PS_OBS_DIM = OBS_DIM + N_AGENTS   # 4 + 2 = 6 (obs + one-hot agent ID)

AGENT_IDS = [
    [1.0, 0.0],  # agent 0 one-hot
    [0.0, 1.0],  # agent 1 one-hot
]


def add_agent_id(obs, agent_idx):
    return obs + AGENT_IDS[agent_idx]


def run_episode_shared(env, shared_policy, max_steps=50, train=True,
                       optimizer=None):
    """
    Run one episode with a single shared policy, agent ID appended.
    Both agents contribute to the same gradient update.
    """
    obs0, obs1 = env.reset()
    tr = {"states": [], "actions": [], "rewards": []}

    for _ in range(max_steps):
        obs0_id = add_agent_id(obs0, 0)
        obs1_id = add_agent_id(obs1, 1)

        with torch.no_grad():
            a0 = Categorical(logits=shared_policy(torch.FloatTensor(obs0_id))).sample().item()
            a1 = Categorical(logits=shared_policy(torch.FloatTensor(obs1_id))).sample().item()

        next_obs0, next_obs1, r0, r1, done, _ = env.step(a0, a1)

        # Both agents' transitions go into the same buffer
        tr["states"].append(obs0_id);  tr["actions"].append(a0); tr["rewards"].append(r0)
        tr["states"].append(obs1_id);  tr["actions"].append(a1); tr["rewards"].append(r1)

        obs0, obs1 = next_obs0, next_obs1
        if done:
            break

    if train and optimizer:
        reinforce_update(shared_policy, optimizer,
                         tr["states"], tr["actions"],
                         discounted_returns(tr["rewards"]))

    joint_reward = sum(tr["rewards"])
    return joint_reward


# ----- Train with parameter sharing ----------------------------------------
torch.manual_seed(0)
shared_policy = PolicyMLP(PS_OBS_DIM, HIDDEN, N_COOP_ACTIONS)
opt_shared    = optim.Adam(shared_policy.parameters(), lr=1e-3)

ps_joint = []
for ep in range(N_EPISODES):
    jr = run_episode_shared(env_coop, shared_policy, optimizer=opt_shared)
    ps_joint.append(jr)

print(f"Parameter sharing    -- last 50 ep joint reward: "
      f"{np.mean(ps_joint[-50:]):.3f}")

## Communication Channel

Before acting, each agent broadcasts a 4-dimensional message vector.
The message is produced by a small "message encoder" network from the agent's
own observation. Each agent then conditions its policy on its own observation
plus the other agent's message.

This is the simplest possible communication: a single round of broadcast before
each decision step.

In [ ]:
MSG_DIM = 4
COMM_OBS_DIM = OBS_DIM + MSG_DIM   # own obs + received message


class CommAgent(nn.Module):
    """
    Agent with a communication channel.

    message_encoder: obs -> MSG_DIM  (differentiable broadcast)
    policy: [obs | received_msg] -> action logits
    """
    def __init__(self, obs_dim, msg_dim, hidden_dim, act_dim):
        super().__init__()
        self.msg_encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, msg_dim),
        )
        self.policy = nn.Sequential(
            nn.Linear(obs_dim + msg_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def encode_message(self, obs_t):
        return self.msg_encoder(obs_t)

    def act_logits(self, obs_t, received_msg_t):
        combined = torch.cat([obs_t, received_msg_t], dim=-1)
        return self.policy(combined)


def run_episode_comm(env, agent_a, agent_b, max_steps=50, train=True,
                     opt_a=None, opt_b=None):
    """
    Episode with communication: each agent broadcasts before acting.
    """
    obs0, obs1 = env.reset()
    tr0 = {"states": [], "msgs": [], "actions": [], "rewards": []}
    tr1 = {"states": [], "msgs": [], "actions": [], "rewards": []}

    for _ in range(max_steps):
        obs0_t = torch.FloatTensor(obs0)
        obs1_t = torch.FloatTensor(obs1)

        with torch.no_grad():
            msg0 = agent_a.encode_message(obs0_t)  # agent_a broadcasts
            msg1 = agent_b.encode_message(obs1_t)  # agent_b broadcasts

            logits0 = agent_a.act_logits(obs0_t, msg1)  # a receives b's msg
            logits1 = agent_b.act_logits(obs1_t, msg0)  # b receives a's msg

            a0 = Categorical(logits=logits0).sample().item()
            a1 = Categorical(logits=logits1).sample().item()

        next_obs0, next_obs1, r0, r1, done, _ = env.step(a0, a1)

        tr0["states"].append(obs0); tr0["actions"].append(a0); tr0["rewards"].append(r0)
        tr1["states"].append(obs1); tr1["actions"].append(a1); tr1["rewards"].append(r1)

        obs0, obs1 = next_obs0, next_obs1
        if done:
            break

    if train and opt_a and opt_b:
        # Full forward pass for gradients (replay the episode)
        def update_comm(agent, tr, opt, other_agent, other_obs_list):
            obs_t  = torch.FloatTensor(tr["states"])
            act_t  = torch.LongTensor(tr["actions"])
            ret_t  = torch.FloatTensor(discounted_returns(tr["rewards"]))
            ret_t  = (ret_t - ret_t.mean()) / (ret_t.std() + 1e-8)
            other_t = torch.FloatTensor(other_obs_list)
            with torch.no_grad():
                other_msgs = other_agent.encode_message(other_t)
            logits = agent.act_logits(obs_t, other_msgs)
            dist   = Categorical(logits=logits)
            loss   = -(dist.log_prob(act_t) * ret_t).mean()
            opt.zero_grad(); loss.backward(); opt.step()

        update_comm(agent_a, tr0, opt_a, agent_b, tr1["states"])
        update_comm(agent_b, tr1, opt_b, agent_a, tr0["states"])

    return sum(tr0["rewards"]), sum(tr1["rewards"])


# ----- Train communication agents -------------------------------------------
torch.manual_seed(0)
comm_a = CommAgent(OBS_DIM, MSG_DIM, HIDDEN, N_COOP_ACTIONS)
comm_b = CommAgent(OBS_DIM, MSG_DIM, HIDDEN, N_COOP_ACTIONS)
opt_ca = optim.Adam(comm_a.parameters(), lr=1e-3)
opt_cb = optim.Adam(comm_b.parameters(), lr=1e-3)

comm_rewards_0, comm_rewards_1, comm_joint = [], [], []
for ep in range(N_EPISODES):
    r0, r1 = run_episode_comm(env_coop, comm_a, comm_b,
                              opt_a=opt_ca, opt_b=opt_cb)
    comm_rewards_0.append(r0)
    comm_rewards_1.append(r1)
    comm_joint.append(r0 + r1)

print(f"Comm agents          -- last 50 ep joint reward: "
      f"{np.mean(comm_joint[-50:]):.3f}")

## Non-Stationarity Visualisation

As each agent improves its policy, the "environment" seen by the other agent
changes. This non-stationarity makes training curves noisier than single-agent RL.

Below we track each agent's individual reward smoothed over a rolling window.
Notice how periods where one agent improves can temporarily hurt the other.

In [ ]:
def smooth(values, window=20):
    return [np.mean(values[max(0, i-window):i+1]) for i in range(len(values))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Independent learners: per-agent reward
ax = axes[0]
ax.plot(smooth(ind_rewards_0), label="Agent 0", color="tab:blue")
ax.plot(smooth(ind_rewards_1), label="Agent 1", color="tab:orange")
ax.set_title("Independent Learners: per-agent reward (smoothed)")
ax.set_xlabel("Episode")
ax.set_ylabel("Episode reward")
ax.legend()
ax.grid(True, alpha=0.3)

# Communication agents: per-agent reward
ax = axes[1]
ax.plot(smooth(comm_rewards_0), label="Agent 0 (comm)", color="tab:blue")
ax.plot(smooth(comm_rewards_1), label="Agent 1 (comm)", color="tab:orange")
ax.set_title("Communication Agents: per-agent reward (smoothed)")
ax.set_xlabel("Episode")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Reward Shaping for Cooperation

Pure individual rewards can lead agents to ignore each other.
Adding a **shared coverage bonus** (reward for both agents having reached their
respective goals together) encourages coordination.

We compare:
- Individual rewards only (already done above as independent learners)
- Individual rewards + a cooperative bonus when both agents succeed together

In [ ]:
COOP_BONUS = 2.0   # extra reward shared when both reach their goals in the same episode


def run_episode_shaped(env, policy_0, policy_1, max_steps=50,
                       opt0=None, opt1=None, coop_bonus=COOP_BONUS):
    """
    Episode with cooperative reward shaping.
    At episode end, if both agents reached their goals, add coop_bonus to both.
    """
    obs0, obs1 = env.reset()
    tr0 = {"states": [], "actions": [], "rewards": []}
    tr1 = {"states": [], "actions": [], "rewards": []}

    both_reached = False
    for _ in range(max_steps):
        with torch.no_grad():
            a0 = Categorical(logits=policy_0(torch.FloatTensor(obs0))).sample().item()
            a1 = Categorical(logits=policy_1(torch.FloatTensor(obs1))).sample().item()

        next_obs0, next_obs1, r0, r1, done, _ = env.step(a0, a1)

        tr0["states"].append(obs0); tr0["actions"].append(a0); tr0["rewards"].append(r0)
        tr1["states"].append(obs1); tr1["actions"].append(a1); tr1["rewards"].append(r1)

        obs0, obs1 = next_obs0, next_obs1
        if done:
            both_reached = True
            break

    # Cooperative shaping: bonus added to the last step's reward
    if both_reached and coop_bonus > 0:
        tr0["rewards"][-1] += coop_bonus
        tr1["rewards"][-1] += coop_bonus

    if opt0 and opt1:
        reinforce_update(policy_0, opt0, tr0["states"], tr0["actions"],
                         discounted_returns(tr0["rewards"]))
        reinforce_update(policy_1, opt1, tr1["states"], tr1["actions"],
                         discounted_returns(tr1["rewards"]))

    return sum(tr0["rewards"]), sum(tr1["rewards"]), both_reached


# Train with cooperative shaping
torch.manual_seed(0)
policy_coop_0 = PolicyMLP(OBS_DIM, HIDDEN, N_COOP_ACTIONS)
policy_coop_1 = PolicyMLP(OBS_DIM, HIDDEN, N_COOP_ACTIONS)
opt_coop_0    = optim.Adam(policy_coop_0.parameters(), lr=1e-3)
opt_coop_1    = optim.Adam(policy_coop_1.parameters(), lr=1e-3)

coop_joint, coop_success = [], []
for ep in range(N_EPISODES):
    r0, r1, success = run_episode_shaped(
        env_coop, policy_coop_0, policy_coop_1,
        opt0=opt_coop_0, opt1=opt_coop_1)
    coop_joint.append(r0 + r1)
    coop_success.append(float(success))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(smooth(ind_joint),  label="Independent (no shaping)", color="tab:red")
ax.plot(smooth(coop_joint), label="Shaped (coop bonus)",      color="tab:green")
ax.set_xlabel("Episode")
ax.set_ylabel("Joint reward (smoothed, window=20)")
ax.set_title("Reward shaping: cooperative bonus vs. pure individual rewards")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Shaped -- last 50 ep success rate:    {np.mean(coop_success[-50:])*100:.1f}%")
print(f"Shaped -- last 50 ep joint reward:    {np.mean(coop_joint[-50:]):.3f}")

## QMIX Simplified: Monotonic Mixing Network

QMIX (Rashid et al., 2018) trains individual Q-networks for each agent but
mixes their Q-values through a **monotonic** hypernetwork to produce a joint
Q-total. Monotonicity means:

```
dQ_tot / dQ_i >= 0   for all i
```

This is the **Individual-Global-Max (IGM)** condition: the joint greedy action
is consistent with each agent independently maximising its own Q-value.

The simplest implementation: use non-negative weights (softplus-activated) to
combine individual Q-values into Q-total.

In [ ]:
class IndividualQNet(nn.Module):
    """Per-agent Q-network: obs -> Q-values for each action."""
    def __init__(self, obs_dim, hidden_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, obs):
        return self.net(obs)   # [B, act_dim]


class MonotonicMixer(nn.Module):
    """
    Simplified QMIX mixing network.

    Takes a vector of per-agent Q-values (the Q of the chosen action for each agent)
    and mixes them into Q_tot with non-negative weights (guarantees monotonicity).

    w = softplus(W_h * state)   -- state-dependent non-negative weights
    Q_tot = sum_i w_i * Q_i + b(state)
    """
    def __init__(self, n_agents, state_dim, mixing_hidden=32):
        super().__init__()
        self.n_agents = n_agents
        # Hypernetwork producing non-negative weights
        self.weight_net = nn.Sequential(
            nn.Linear(state_dim, mixing_hidden),
            nn.ReLU(),
            nn.Linear(mixing_hidden, n_agents),
        )
        # Bias network (can be negative)
        self.bias_net = nn.Sequential(
            nn.Linear(state_dim, mixing_hidden),
            nn.ReLU(),
            nn.Linear(mixing_hidden, 1),
        )

    def forward(self, q_values, states):
        """
        q_values : [B, n_agents]  -- one Q-value per agent
        states   : [B, state_dim] -- global state used by hypernetwork

        Returns Q_tot : [B]
        """
        # Non-negative weights via softplus
        w = torch.nn.functional.softplus(self.weight_net(states))  # [B, n_agents]
        b = self.bias_net(states).squeeze(-1)                       # [B]
        q_tot = (w * q_values).sum(dim=-1) + b                     # [B]
        return q_tot


# ---- Verify monotonicity ---------------------------------------------------
n_agents  = 2
state_dim = OBS_DIM * n_agents   # concatenated observations as global state

mixer = MonotonicMixer(n_agents=n_agents, state_dim=state_dim)
q_ind = torch.FloatTensor([[0.5, 0.3]])   # [1, 2]: Q for agent0, agent1
state = torch.zeros(1, state_dim)

q_tot_base = mixer(q_ind, state).item()

# Increase agent0's Q by 0.1 -- Q_tot should also increase
q_higher = q_ind.clone(); q_higher[0, 0] += 0.1
q_tot_higher = mixer(q_higher, state).item()

print(f"Q_individual = {q_ind.tolist()}")
print(f"Q_tot (base) = {q_tot_base:.4f}")
print(f"Q_individual = {q_higher.tolist()} (agent0 +0.1)")
print(f"Q_tot (higher) = {q_tot_higher:.4f}")
print(f"Monotonic increase: {q_tot_higher > q_tot_base}  (expected True)")

# Show that all partial derivatives dQ_tot/dQ_i >= 0
q_grad = q_ind.clone().requires_grad_(True)
q_tot  = mixer(q_grad, state)
q_tot.backward()
grads = q_grad.grad[0]
print(f"\ndQ_tot/dQ_0 = {grads[0].item():.4f}  (>= 0? {grads[0].item() >= 0})")
print(f"dQ_tot/dQ_1 = {grads[1].item():.4f}  (>= 0? {grads[1].item() >= 0})")

## Training Metrics: Per-Agent, Joint Reward, and Cooperation Metric

We plot all three training runs together and add a cooperation metric:
the sum of Manhattan distances between both agents and their respective goals
at the end of each episode (lower = better coordination).

In [ ]:
def eval_cooperation_metric(env, policy_0, policy_1, n_episodes=50, max_steps=50):
    """
    Measure average end-of-episode distance between agents and their goals.
    Lower values indicate better coordination.
    """
    dist_log = []
    for _ in range(n_episodes):
        obs0, obs1 = env.reset()
        final_d = (abs(env._pos[0][0] - env.goal_0[0]) + abs(env._pos[0][1] - env.goal_0[1]) +
                   abs(env._pos[1][0] - env.goal_1[0]) + abs(env._pos[1][1] - env.goal_1[1]))
        for _ in range(max_steps):
            with torch.no_grad():
                a0 = Categorical(logits=policy_0(torch.FloatTensor(obs0))).sample().item()
                a1 = Categorical(logits=policy_1(torch.FloatTensor(obs1))).sample().item()
            obs0, obs1, _, _, done, info = env.step(a0, a1)
            final_d = info["d0"] + info["d1"]
            if done:
                break
        dist_log.append(final_d)
    return np.mean(dist_log)


coop_metric_ind   = eval_cooperation_metric(env_coop, policy_ind_0, policy_ind_1)
coop_metric_coop  = eval_cooperation_metric(env_coop, policy_coop_0, policy_coop_1)
coop_metric_comm  = eval_cooperation_metric(env_coop,
    lambda obs: policy_ind_0(torch.FloatTensor(obs)),   # using comm_a as proxy
    lambda obs: policy_ind_1(torch.FloatTensor(obs)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Joint reward comparison
ax = axes[0]
ax.plot(smooth(ind_joint),   label="Independent",    color="tab:red")
ax.plot(smooth(ps_joint),    label="Param sharing",  color="tab:purple")
ax.plot(smooth(coop_joint),  label="Coop shaping",   color="tab:green")
ax.plot(smooth(comm_joint),  label="Comm channel",   color="tab:blue")
ax.set_title("Joint reward (all methods)")
ax.set_xlabel("Episode")
ax.set_ylabel("Joint reward (smoothed)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Per-agent reward for independent learners
ax = axes[1]
ax.plot(smooth(ind_rewards_0), label="Agent 0", color="tab:blue")
ax.plot(smooth(ind_rewards_1), label="Agent 1", color="tab:orange")
ax.set_title("Independent Learners: per-agent reward")
ax.set_xlabel("Episode")
ax.legend()
ax.grid(True, alpha=0.3)

# Cooperation metric bar chart
ax = axes[2]
methods  = ["Independent", "Coop shaping"]
metrics  = [coop_metric_ind, coop_metric_coop]
colors   = ["tab:red", "tab:green"]
ax.bar(methods, metrics, color=colors)
ax.set_ylabel("Avg end-of-ep goal distance (lower=better)")
ax.set_title("Cooperation metric (distance to goals)")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Exercise: Add a Third Agent

The parameter-sharing approach scales to N agents with zero code changes: just
extend the one-hot agent ID to length 3 and add a third start/goal pair.
Independent learning requires a new policy network and optimizer for each agent added.

Extend the `CoopGridWorld` to handle 3 agents, update the parameter-sharing
training loop, and compare joint reward for N=2 vs N=3 using the same number of
training episodes.

In [ ]:
class CoopGridWorld3(CoopGridWorld):
    """
    Extend CoopGridWorld to 3 agents.
    Agent 2 starts at (COOP_SIZE//2, 0) and must reach goal_2.

    Observation for agent i: [r_self, c_self, r_a1, c_a1, r_a2, c_a2]
    (6-dimensional, other agents listed in fixed order excluding self)
    """

    def __init__(self, size=COOP_SIZE,
                 goal_0=(5, 5), goal_1=(5, 0), goal_2=(5, 2)):
        super().__init__(size, goal_0, goal_1)
        self.goal_2   = goal_2
        self._pos     = [None, None, None]
        self._at_goal = [False, False, False]

    def reset(self):
        self._pos     = [(0, 0), (0, self.size - 1), (0, self.size // 2)]
        self._at_goal = [False, False, False]
        return self._obs3()

    def _obs3(self):
        """Return obs for each of 3 agents."""
        positions = self._pos
        goals     = [self.goal_0, self.goal_1, self.goal_2]
        obs = []
        for i in range(3):
            ri, ci = positions[i]
            own = [ri / (self.size-1), ci / (self.size-1)]
            others = []
            for j in range(3):
                if j != i:
                    rj, cj = positions[j]
                    others += [rj / (self.size-1), cj / (self.size-1)]
            obs.append(own + others)   # length 6
        return obs   # list of 3 obs vectors

    def step3(self, actions):
        """
        actions : list of 3 action indices

        Returns obs_list, rewards, done, info.
        """
        # YOUR CODE HERE
        raise NotImplementedError


# ----- Parameter-sharing training with 3 agents ----------------------------
# YOUR CODE HERE
# 1. Instantiate CoopGridWorld3.
# 2. Create a shared PolicyMLP with obs_dim=6+3=9 (obs + 3-dim one-hot ID).
# 3. Run N_EPISODES episodes with all 3 agents using the shared policy.
# 4. Plot joint reward for N=2 (parameter sharing, already computed above) and N=3.

raise NotImplementedError

## Practical Considerations for the Competition

**Observation space.** Does each agent observe the full grid or only what's
within some radius? Partial observability makes the problem harder and changes
what information the policy can use.

**Shared vs. separate policies.** In symmetric games where all agents have the same
role and capabilities, a single shared policy is more sample-efficient.
In asymmetric games, separate policies are necessary.

**Reward shaping.** Sparse reward (only +1/-1 at the end) is harder to learn
from than shaped rewards (small positive for moving closer to goal).
Be careful: shaping rewards can introduce unintended behaviors.

**Evaluation.** Win rate against a fixed baseline (an older version of your own policy,
or a hand-crafted heuristic) is more informative than win rate against a random opponent.
It's easy to overfit to a weak fixed opponent and regress when the opponent improves.